<a href="https://colab.research.google.com/github/cpython-projects/python_da_17_03_26/blob/main/lesson_19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import files

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('kc_house_data.csv')
df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180.0,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170.0,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770.0,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050.0,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680.0,0,1987,0,98074,47.6168,-122.045,1800,7503


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21613 non-null  int64  
 1   date           21613 non-null  object 
 2   price          21613 non-null  float64
 3   bedrooms       21613 non-null  int64  
 4   bathrooms      21613 non-null  float64
 5   sqft_living    21613 non-null  int64  
 6   sqft_lot       21613 non-null  int64  
 7   floors         21613 non-null  float64
 8   waterfront     21613 non-null  int64  
 9   view           21613 non-null  int64  
 10  condition      21613 non-null  int64  
 11  grade          21613 non-null  int64  
 12  sqft_above     21611 non-null  float64
 13  sqft_basement  21613 non-null  int64  
 14  yr_built       21613 non-null  int64  
 15  yr_renovated   21613 non-null  int64  
 16  zipcode        21613 non-null  int64  
 17  lat            21613 non-null  float64
 18  long  

# Групування даних з `groupby()`

## Що таке `groupby()`

Метод `groupby()` дозволяє згрупувати дані за одним або кількома критеріями для подальшого узагальнення (агрегації).

**Загальна формула:**

```python
df.groupby(ключ)[поля].агрегуюча_функція()
```

## Сценарії використання

In [ ]:
# Групування по одному полю — одна метрика


In [ ]:
# Групування по одному полю — кілька метрик


In [ ]:
# Групування по кількох полях


In [ ]:
# Агрування кількох полів — кілька метрик


## `reset_index()`

Після `groupby()` індексом стає групувальне поле (або поля). Щоб зробити з них **звичайні колонки**, потрібно:

```python
.reset_index()
```

Інакше графіки чи подальша робота будуть ускладнені.

## Задача: обчислити:

* середню ціну (`price`)
* кількість оголошень (`count`)
* медіанну площу (`sqft_living`)
  по кожному району

# Агрегації `.agg()` в Pandas

## Навіщо потрібні агрегації?

Агрегація — це зведення (узагальнення) числових значень:

| Приклади запитань                      | Агрегація  |
| -------------------------------------- | ---------- |
| Яка середня площа будинків в районі Х?     | `mean()`   |
| Скільки будинків виставлено у Y? | `count()`  |
| Яка максимальна ціна у 1 bedrooms?         | `max()`    |
| Яка медіанна кухня у 3-bedrooms?      | `median()` |

---

## Синтаксис `.agg()` — на одному або кількох полях

```python
df.agg({'col1': 'mean', 'col2': 'max'})
```

Або на одній колонці:

```python
df['price'].agg(['mean', 'median', 'std'])
```

---

## Комбінування з `groupby()`

```python
df.groupby('zipcode').agg({
    'price': ['mean', 'median', 'min', 'max'],
    'sqft_living': 'mean'
})
```

У результаті — **ієрархічний заголовок колонок (MultiIndex)** → бажано `.reset_index()` та перейменувати колонки.

In [ ]:
summary = df[['price', 'sqft_living']].agg(
    ['mean', 'median', 'std', 'min', 'max', 'count']
).round(1).reset_index()

summary


,index,price,sqft_living
0,mean,540088.1,2079.9
1,median,450000.0,1910.0
2,std,367127.2,918.4
3,min,75000.0,290.0
4,max,7700000.0,13540.0
5,count,21613.0,21613.0


In [ ]:
# T - transpose


## Нюанси

| Питання                       | Коментар                                     |
| ----------------------------- | -------------------------------------------- |
| `NaN`                         | Ігноруються у функціях агрегування           |
| `.agg()` після `groupby()`    | Повертає MultiIndex → `.reset_index()`       |
| `std`, `min`, `max`, `median` | Обчислюються лише для числових               |
| Для bool                      | Можна використовувати `mean()` — доля `True` |

---

# Зведені таблиці (`pivot_table`)

## Що таке `pivot_table`

`pivot_table()` — це розширений варіант `groupby`, який дозволяє:

* Зробити зведену таблицю по кількох категоріях
* Розмістити одну категорію по рядках, іншу по стовпцях
* Одразу застосувати агрегуючу функцію
* Аналог Excel PivotTable!

---

## Синтаксис

```python
pd.pivot_table(
    data=df,
    values='метрика',
    index='рядки',
    columns='стовпці',
    aggfunc='функція'
)
```

In [ ]:
# Приклад: середня ціна по містах і кількості кімнат


## Додаткові параметри

| Параметр                     | Значення            |
| ---------------------------- | ------------------- |
| `fill_value`                 | чим заповнити `NaN` |
| `margins=True`               | додати підсумки     |
| `aggfunc=['mean', 'median']` | кілька функцій      |


# Частотні таблиці — `pd.crosstab()`

## Що таке `crosstab`

`pd.crosstab()` — це спосіб підрахунку **кількості випадків** для комбінацій категоріальних змінних.

## Синтаксис

```python
pd.crosstab(index=df['рядки'], columns=df['стовпці'])
```

In [ ]:
df.columns.to_list()

['id',
 'price',
 'bedrooms',
 'bathrooms',
 'sqft_living',
 'sqft_lot',
 'floors',
 'waterfront',
 'view',
 'condition',
 'grade',
 'sqft_above',
 'sqft_basement',
 'yr_built',
 'yr_renovated',
 'zipcode',
 'lat',
 'long',
 'sqft_living15',
 'sqft_lot15',
 'area_category',
 'area_auto_category']

In [ ]:
# Приклад: скільки якісних будинків було оновлено


## Нюанси

| Питання                            | Коментар                              |
| ---------------------------------- | ------------------------------------- |
| `NaN` у категоріях                 | не враховуються                       |
| `normalize='index'`                | показує **частки** замість кількостей |
| Можна додати `values=`, `aggfunc=` | як у pivot\_table                     |

---

In [ ]:
# Тип житла × ціновий сегмент


# `resample()` — агрегування за датами

## Що таке `resample`

`resample()` — це аналог `groupby()` **по часу**, який працює з колонкою `datetime`.

Необхідно:

* мати колонку типу `datetime64[ns]`
* встановити її як **індекс**

---

## Синтаксис:

```python
df.resample('період')['метрика'].agg(функція)
```


### Базові часові одиниці

| Частота     | Alias             | Опис             |
| ----------- | ----------------- | ---------------- |
| Day         | `'D'`             | календарний день |
| Hour        | `'H'` або `'h'`   | година           |
| Minute      | `'T'` або `'min'` | хвилина          |
| Second      | `'S'` або `'s'`   | секунда          |
| Millisecond | `'ms'`            | мілісекунда      |
| Microsecond | `'us'`            | мікросекунда     |
| Nanosecond  | `'ns'`            | наносекунда      |

---

### Бізнес-час і бізнес-дні

| Частота                  | Alias   | Опис                                                 |
| ------------------------ | ------- | ---------------------------------------------------- |
| BDay / BusinessDay       | `'B'`   | бізнес-день (пн–пт)                                  |
| CDay / CustomBusinessDay | `'C'`   | кастомний бізнес-день (за користувацьким календарем) |
| BusinessHour             | `'bh'`  | бізнес-година                                        |
| CustomBusinessHour       | `'cbh'` | кастомна бізнес-година                               |

---

### Тижні

| Частота         | Alias    | Опис                                           |
| --------------- | -------- | ---------------------------------------------- |
| Week            | `'W'`    | тиждень, можна якір на день (`W-MON`, `W-FRI`) |
**Приклади**

* `'W'` — тиждень, кінець у неділю
* `'W-MON'` — тиждень з кінцем у понеділок

---

### Місячні частоти

| Частота        | Alias              | Опис                            |
| -------------- | ------------------ | ------------------------------- |
| MonthEnd       | `'M'` або `'ME'`   | кінець календарного місяця      |
| MonthBegin     | `'MS'`             | початок місяця                  |
| BMonthEnd      | `'BM'` або `'BME'` | бізнес-кінець місяця            |
| BMonthBegin    | `'BMS'`            | бізнес-початок місяця           |

---

### Квартали

| Частота       | Alias              | Опис                                    |
| ------------- | ------------------ | --------------------------------------- |
| QuarterEnd    | `'Q'` або `'QE'`   | кінець календарного кварталу            |
| QuarterBegin  | `'QS'`             | початок кварталу                        |
| BQuarterEnd   | `'BQ'` або `'BQE'` | бізнес-кінець кварталу                  |
| BQuarterBegin | `'BQS'`            | бізнес-початок кварталу                 |

**Приклади**

* `'Q-MAR'` — квартал, що закінчується у березні
* `'QS-APR'` — квартал починається у квітні

---

### Річні частоти

| Частота    | Alias                   | Опис                       |
| ---------- | ----------------------- | -------------------------- |
| YearEnd    | `'A'` або `'YE'`        | кінець календарного року   |
| YearBegin  | `'AS'`, `'YS'`, `'BYS'` | початок року               |
| BYearEnd   | `'BA'`, `'BYE'`         | бізнес-кінець року         |
| BYearBegin | `'BAS'`, `'BYS'`        | бізнес-початок року        |


**Приклади**

* `'A-DEC'` — рік закінчується у грудні
* `'AS-JUL'` — рік починається у липні

In [ ]:
# Підготовка даних
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             21613 non-null  int64         
 1   date           0 non-null      datetime64[ns]
 2   price          21613 non-null  float64       
 3   bedrooms       21613 non-null  int64         
 4   bathrooms      21613 non-null  float64       
 5   sqft_living    21613 non-null  int64         
 6   sqft_lot       21613 non-null  int64         
 7   floors         21613 non-null  float64       
 8   waterfront     21613 non-null  int64         
 9   view           21613 non-null  int64         
 10  condition      21613 non-null  int64         
 11  grade          21613 non-null  int64         
 12  sqft_above     21611 non-null  float64       
 13  sqft_basement  21613 non-null  int64         
 14  yr_built       21613 non-null  int64         
 15  yr_renovated   2161

In [ ]:
# Приклад: середня ціна будинка по місяцях


In [ ]:
# Візуалізація: динаміка середньої ціни по місяцях


In [ ]:
# Кількість угод по тижнях


In [ ]:
# Візуалізація: кількість угод по тижнях


## Навіщо аналітику `resample`

| Що дізнаємось                         | Як це корисно                       |
| ------------------------------------- | ----------------------------------- |
| Сезонність ринку                      | порівняння цін по місяцях/роках     |
| Активність користувачів               | по кількості оголошень              |
| Аналіз впливу подій (інфляція, війна) | тренди по кварталах                 |
| Маркетинг                             | коли зростає інтерес до нерухомості |

---

## Нюанси

| Проблема          | Рішення                                 |
| ----------------- | --------------------------------------- |
| Не дата в індексі | `.set_index('datetime')`                |
| Пропущені періоди | Автоматично створюються `NaN`           |
| Смугастий графік  | Використовуйте `'M'`, `'Q'`, а не `'D'` |
| Великі значення   | Обрізати викиди перед побудовою         |

---

# `pd.cut()` — групування числових значень у категорії

## Що таке `cut`

`cut()` розбиває числову колонку на **інтервали (бінінг)**. Наприклад, можна створити категорії будинків:

* до 40 м² → «малий»
* 40–80 м² → «середній»
* понад 80 м² → «великий»

Це корисно для:

* **аналізу розподілу** (гістограми, pie charts)
* **порівнянь середніх цін між групами**
* **створення сегментів**

---

## Синтаксис

```python
pd.cut(Series, bins, labels=..., include_lowest=True)
```

| Параметр         | Значення                         |
| ---------------- | -------------------------------- |
| `bins`           | кількість або список меж         |
| `labels`         | імена категорій                  |
| `include_lowest` | включити нижню межу в перший бін |

---

In [ ]:
# Приклад: Розбиття будитнків за площею


In [ ]:
# Альтернатива: qcut() — автоматичне розбиття на квантилі
# Корисно, коли хочемо однакову кількість спостережень у групі, а не однакову ширину інтервалу.



## Навіщо аналітику `cut` і `qcut`

| Задача                             | Рішення                |
| ---------------------------------- | ---------------------- |
| Побудова сегментів                 | `cut` або `qcut`       |
| Когортний аналіз (покоління, ціна) | `cut` по даті або ціні |
| Спрощення числових фіч             | категоризація          |
| Візуалізація (pie, bar, boxplot)   | категорії зручніші     |

---

## Типові помилки

| Проблема                          | Рішення                                   |
| --------------------------------- | ----------------------------------------- |
| Значення на межі не включене      | `include_lowest=True`                     |
| Некоректні категорії              | задати `labels` вручну або перевірити     |
| Багато унікальних значень у `cut` | замість `cut` — використовувати `qcut`    |
| Нерівномірні категорії            | вручну задати `bins`, якщо треба контроль |

---